# Report Generation — Stage B (score with RadEval)

Score a pairs parquet produced by
[`report_generation_example.ipynb`](report_generation_example.ipynb)
(Stage A / `ReportGenerationEvaluator.generate_only`) with the
[RadEval](https://github.com/jbdel/RadEval) metric suite.

This notebook is **deliberately standalone** — it imports only `radeval` +
`pandas` (not `radharmony`, `torch`, or `transformers`) so it runs in the
RadEval env, fully decoupled from the generation env. Input is the parquet
with columns `sample_id, reference, hypothesis`.

**Install (in its own venv):**
```bash
uv venv --python 3.12 .venv-radeval
uv pip install --python .venv-radeval/bin/python \
  "radeval[api] @ git+https://github.com/jbdel/RadEval.git" pandas pyarrow
```

## Configuration

In [ ]:
PAIRS_PARQUET  = "outputs/reportgen/pairs.parquet"
OUTPUT_DIR     = "outputs/reportgen"
LIGHT          = False   # True -> LIGHT_METRICS (no LLM/API) instead of the full 16
N_BOOTSTRAP    = 0
BOOTSTRAP_SEED = 0
NORMALIZE_HYP  = True    # strip CheXagent grounded-tags/bold before scoring

## Hypothesis normalization

CheXagent-2 emits two native grounded formats:

- `[Region: Subregion] **finding**` (from DICOM input)
- `<|ref|> finding <|/ref|> <|box|> (x,y),(x,y) <|/box|>` (from PNG input)

Both tank BLEU/ROUGE/F1-CheXbert against free-text references. Strip the
structural markup before scoring so we compare **content**, not format.
Set `NORMALIZE_HYP = False` to score raw model output.

In [ ]:
import re

_GROUNDED_TAG = re.compile(r"\[[A-Za-z][A-Za-z _/&\-]+:\s*[A-Za-z][A-Za-z _/&\-]+\]\s*")
_REF_OPEN  = re.compile(r"<\|ref\|>\s*")
_REF_CLOSE = re.compile(r"\s*<\|/ref\|>")
_BOX       = re.compile(r"<\|box\|>.*?<\|/box\|>", re.DOTALL)
_BOLD_MARK = re.compile(r"\*+")
_WS        = re.compile(r"\s+")


def normalize_hypothesis(s: str) -> str:
    s = _GROUNDED_TAG.sub("", s)
    s = _BOX.sub("", s)
    s = _REF_OPEN.sub("", s)
    s = _REF_CLOSE.sub("", s)
    s = _BOLD_MARK.sub("", s)
    s = _WS.sub(" ", s).strip()
    return s

## Metric bundles and result flattener

RadEval returns a nested dict (some metrics emit sub-scores, e.g.
`radgraph -> {f1, precision, recall}`); `_flatten` collapses it to
`{column: float}`.

In [ ]:
import numpy as np

FULL_METRICS = (
    "bleu", "rouge", "bertscore", "radeval_bertscore", "f1chexbert",
    "f1radbert_ct", "radgraph", "ratescore", "radgraph_radcliq", "radcliq",
    "srrbert", "temporal", "green", "mammo_green", "crimson", "radfact_ct",
)
LIGHT_METRICS = ("bleu", "rouge", "bertscore", "radgraph", "f1chexbert")


def _flatten(d: dict, prefix: str = "") -> dict:
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(_flatten(v, prefix=f"{key}_"))
        elif isinstance(v, bool):
            continue
        elif isinstance(v, (int, float, np.floating)):
            out[key] = float(v)
    return out

## Load pairs and normalize

In [ ]:
import pandas as pd

df = pd.read_parquet(PAIRS_PARQUET)
refs = df["reference"].astype(str).tolist()
hyps_raw = df["hypothesis"].astype(str).tolist()

if NORMALIZE_HYP:
    hyps = [normalize_hypothesis(h) for h in hyps_raw]
    n_changed = sum(1 for a, b in zip(hyps_raw, hyps) if a != b)
    print(f"hyp normalize: stripped grounded tags/bold from {n_changed}/{len(hyps)} samples")
else:
    hyps = hyps_raw

metrics = list(LIGHT_METRICS if LIGHT else FULL_METRICS)
print(f"scoring {len(refs)} pairs with {len(metrics)} metrics: {metrics}")

## Score with RadEval

`radeval` is the only heavy import and is loaded lazily here. Bootstrap
resampling (if `N_BOOTSTRAP > 0`) re-runs every metric on a bootstrap draw of
the pairs for confidence intervals.

In [ ]:
from radeval import RadEval  # lazy: the only heavy import

evaluator = RadEval(metrics=metrics)


def _score(r, h):
    return _flatten(evaluator(refs=r, hyps=h))


rows = [{"label": "report", "bootstrap": -1, **_score(refs, hyps)}]
if N_BOOTSTRAP > 0:
    rng = np.random.default_rng(BOOTSTRAP_SEED & 0xFFFFFFFF)
    n = len(refs)
    for b in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        rows.append({
            "label": "report", "bootstrap": b,
            **_score([refs[i] for i in idx], [hyps[i] for i in idx]),
        })

## Write results

In [ ]:
import json, os

os.makedirs(OUTPUT_DIR, exist_ok=True)
out = pd.DataFrame(rows)
out.to_csv(os.path.join(OUTPUT_DIR, "results.csv"), index=False)
point = {k: v for k, v in rows[0].items() if k not in ("label", "bootstrap")}
with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump({"n_samples": len(refs), "metrics": point}, f, indent=2)
print(f"n={len(refs)}  results -> {OUTPUT_DIR}/")
print(json.dumps(point, indent=2))